# Cochleogram-ViT — AST (AudioSet-pretrained ViT)

The report's Tier-1 lever. The previous from-scratch ViT (notebook 18) and ImageNet-
pretrained ViT (notebooks 09-11) plateau because their pretraining domain doesn't
match audio spectrograms. AST is the same ViT architecture but pretrained on
**AudioSet (2M audio clips) + ImageNet** — the right domain.

## What this notebook does

- Loads `MIT/ast-finetuned-audioset-10-10-0.4593` (DeiT-base AST, ~87M params)
- Interpolates the pretrained positional embeddings from AST's native 1024-time
  to our 128-time cochleograms (preserves the pretrained spatial structure)
- Feeds the cochleogram **as 1-channel** (the report explicitly says drop the
  viridis RGB rendering — it's nonlinear noise that hurts AST)
- Normalizes per-sample to match AST's expected input statistics
- Same proven recipe: leak-free Subset, single-correction weighted CE with
  SOFTEN_POWER=0.25, StratifiedGroupKFold (same seed as nb 18 for direct comparison)
- AdamW lr=5e-5, batch 8 (for VRAM with 87M params on 8GB GPU), 30 epochs
- Saves per-fold softmax probs for ensembling

## Expected
- Mang et al.'s cochleogram + from-scratch ViT: 64.03% (your notebook 18: 63.61%)
- Bae et al.'s AST fine-tune on ICBHI: 59.55% (official 60/40 split)
- With our stratified 10-fold + cochleogram input, realistic target: **65-70%**
- First positive result on top of the methodology fixes

## Important caveats
- First run downloads ~340MB of pretrained weights
- The cochleogram-as-spectrogram-input is well-founded (the report explicitly
  notes "the patch-embedding and transformer are agnostic to whether the 2D input
  is mel or gammatone"); the pretrained AST features should transfer

NOTE: AST uses 1-channel input, so a NEW dataset class is used (no viridis RGB).
The original `cochleograms/` dir is read directly.


In [1]:
# --- Imports + paths ---
import os, json, time, copy, gc, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from transformers import ASTConfig, ASTForAudioClassification

DATA_DIR      = '../data/processed/cochleograms'
METADATA_PATH = '../data/processed/metadata.csv'
RESULTS_PATH  = '../results/ast.json'
PREDS_PATH    = '../results/ast_preds.npz'

# Hyperparams (per AST-on-ICBHI recipe from Bae et al., adjusted)
BATCH_SIZE    = 8           # AST is ~87M; 8GB GPU
EPOCHS        = 30
LEARNING_RATE = 5e-5        # AST fine-tune LR (lower than from-scratch)
WEIGHT_DECAY  = 0.01        # AdamW
SOFTEN_POWER  = 0.25        # best from sweep

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')


device: cuda


In [2]:
# --- AST wrapper: adapt 1024-time pretrained model to 128-time cochleograms ---

class CochleogramAST(nn.Module):
    """
    AudioSet+ImageNet pretrained AST, adapted to (128 freq x 128 time) cochleograms.

    Adapts the model by:
      1. Loading config with our num_mel_bins=128, max_length=128
      2. Loading pretrained weights with ignore_mismatched_sizes=True (skips pos embeddings + head)
      3. Manually interpolating the pretrained positional embeddings from the original
         (12 freq patches x 101 time patches) grid to our (12 x 12) grid via bilinear interp,
         preserving the pretrained spatial structure (the AST paper's standard adaptation).

    Cochleogram input is transposed so the spatial layout matches AST's (time, freq) convention.
    Per-sample normalization standardizes to roughly AST's expected statistics.
    """
    def __init__(self, num_labels=4,
                 ckpt='MIT/ast-finetuned-audioset-10-10-0.4593',
                 target_max_length=128, target_num_mel_bins=128):
        super().__init__()
        self.target_max_length = target_max_length
        self.target_num_mel_bins = target_num_mel_bins

        # ---- 1. config with our target shape ----
        cfg = ASTConfig.from_pretrained(ckpt)
        orig_max_len = cfg.max_length
        orig_mel     = cfg.num_mel_bins
        ps           = cfg.patch_size
        fs, ts       = cfg.frequency_stride, cfg.time_stride
        orig_f_patches = (orig_mel - ps) // fs + 1
        orig_t_patches = (orig_max_len - ps) // ts + 1
        new_f_patches  = (target_num_mel_bins - ps) // fs + 1
        new_t_patches  = (target_max_length    - ps) // ts + 1
        print(f'  pretrained grid: {orig_f_patches} freq x {orig_t_patches} time = {orig_f_patches*orig_t_patches} patches')
        print(f'  target grid:     {new_f_patches} freq x {new_t_patches} time = {new_f_patches*new_t_patches} patches')

        # Build config with our target shape, num_labels=4
        cfg.max_length   = target_max_length
        cfg.num_mel_bins = target_num_mel_bins
        cfg.num_labels   = num_labels

        # ---- 2. load pretrained with mismatched sizes (head + position embeddings) ----
        self.model = ASTForAudioClassification.from_pretrained(
            ckpt, config=cfg, ignore_mismatched_sizes=True,
        )

        # ---- 3. interpolate pretrained positional embeddings ----
        pretrained = ASTForAudioClassification.from_pretrained(ckpt)
        pe_pre = pretrained.audio_spectrogram_transformer.embeddings.position_embeddings.data  # (1, 1+1+N_old, H)
        H = pe_pre.shape[-1]
        # 2 special tokens (CLS + distillation) at the front
        special_tokens = pe_pre[:, :2, :]                                                     # (1, 2, H)
        patch_pe       = pe_pre[:, 2:, :]                                                     # (1, 1212, H)
        patch_pe       = patch_pe.reshape(1, orig_f_patches, orig_t_patches, H).permute(0,3,1,2)  # (1,H,f,t)
        # bilinear interp to (new_f_patches, new_t_patches)
        patch_pe_new   = F.interpolate(patch_pe, size=(new_f_patches, new_t_patches),
                                        mode='bilinear', align_corners=False)
        patch_pe_new   = patch_pe_new.permute(0,2,3,1).reshape(1, new_f_patches*new_t_patches, H)  # (1, N_new, H)
        pe_new         = torch.cat([special_tokens, patch_pe_new], dim=1)
        # assign
        self.model.audio_spectrogram_transformer.embeddings.position_embeddings.data = pe_new.clone()
        del pretrained
        print(f'  position embeddings interpolated: {pe_pre.shape[1]} -> {pe_new.shape[1]} tokens')

        self._log_param_count()

    def forward(self, x):
        """
        x: (B, 128, 128) cochleogram (freq x time, [0,1] range)
           -- normalized + transposed to (B, 128, 128) (time, freq) internally
        Returns: logits (B, num_labels)
        """
        # Cochleogram in our dataset is (freq=128, time=128).
        # AST expects (time=max_length, freq=num_mel_bins) -> transpose.
        x = x.transpose(-1, -2)              # (B, 128, 128) -- now (time, freq)
        # Per-sample normalize (zero-mean unit-std) -- robust default
        m = x.mean(dim=(-1,-2), keepdim=True)
        s = x.std (dim=(-1,-2), keepdim=True).clamp_min(1e-6)
        x = (x - m) / s
        return self.model(input_values=x).logits

    def _log_param_count(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'[CochleogramAST] Parameters - total: {total:,}  trainable: {trainable:,}')


# ---- Smoke test ----
import os
print('Building CochleogramAST (downloads pretrained weights on first run, ~340MB)...')
m = CochleogramAST(num_labels=4)
x = torch.randn(2, 128, 128)  # B=2, freq, time
y = m(x)
assert y.shape == (2, 4), f'unexpected output shape: {y.shape}'
print(f'smoke test OK: output shape {tuple(y.shape)}')
del m


Building CochleogramAST (downloads pretrained weights on first run, ~340MB)...


  pretrained grid: 12 freq x 101 time = 1212 patches
  target grid:     12 freq x 12 time = 144 patches


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 3637.48it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([4, 768])         
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.S

  position embeddings interpolated: 1214 -> 146 tokens
[CochleogramAST] Parameters - total: 85,371,652  trainable: 85,371,652
smoke test OK: output shape (2, 4)


In [3]:
# --- Dataset (1-channel cochleogram) + StratifiedGroupKFold (same seed as nb 18) ---
class CochleogramDataset1ch(Dataset):
    """Returns the raw 1-channel cochleogram (no viridis RGB). Shape: (128, 128)."""
    def __init__(self, data_dir, metadata_path):
        self.data_dir = data_dir
        self.metadata = pd.read_csv(metadata_path)
    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        npy_path = os.path.join(self.data_dir, os.path.basename(row['npy_path']))
        coch = np.load(npy_path).astype(np.float32)  # (128, 128) in [0,1]
        return torch.from_numpy(coch), int(row['label'])

dataset = CochleogramDataset1ch(DATA_DIR, METADATA_PATH)
metadata = dataset.metadata.copy()
metadata['patient_id'] = metadata['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])
print(f'dataset size: {len(dataset)}')

# Same CV strategy + seed as notebook 18 -- IDENTICAL folds for direct comparison
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
FOLDS = list(sgkf.split(metadata, metadata['label'].values, groups=metadata['patient_id'].values))
print(f'StratifiedGroupKFold: {len(FOLDS)} folds (same seed=42 as nb 18)')


dataset size: 6898
StratifiedGroupKFold: 10 folds (same seed=42 as nb 18)


In [4]:
# --- Helpers (same paper-conv metric + softening + lr schedule) ---
def softened_weights(softpow):
    raw = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=metadata['label'].values)
    w = raw ** softpow
    return w / w.sum() * len(w)

def lr_lambda(epoch):
    warmup = 4
    if epoch < warmup:
        return (epoch + 1) / warmup
    denom = EPOCHS - warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / denom)) if denom > 0 else 0.0

def paper_metrics(preds, labels):
    p = np.asarray(preds); y = np.asarray(labels)
    TP   = int(np.sum((y != 0) & (p == y)))
    FN   = int(np.sum((y != 0) & (p == 0)))
    FN_w = int(np.sum((y != 0) & (p != 0) & (p != y)))
    TN   = int(np.sum((y == 0) & (p == 0)))
    FP   = int(np.sum((y == 0) & (p != 0)))
    TP_b = TP + FN_w
    se = TP_b / (TP_b + FN + 1e-8); sp = TN / (TN + FP + 1e-8)
    return {'TP': TP, 'FN': FN, 'FN_wrong': FN_w, 'TN': TN, 'FP': FP,
            'se': float(se), 'sp': float(sp), 'score': float((se + sp) / 2)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        labels.extend(y.numpy().tolist())
    return preds, labels, np.concatenate(probs, axis=0)


In [5]:
# --- Training: 10-fold StratifiedGroupKFold ---
cw = softened_weights(SOFTEN_POWER)
cw_t = torch.tensor(cw, dtype=torch.float).to(device)
print(f'class weights (^{SOFTEN_POWER}): {np.round(cw, 3).tolist()}')

fold_rows, pooled_preds, pooled_labels = [], [], []
per_fold_probs, per_fold_labels, per_fold_val_idx = {}, {}, {}
t_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLDS):
    torch.manual_seed(42 + fold); np.random.seed(42 + fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42 + fold)

    train_subset = Subset(dataset, train_idx)
    val_subset   = Subset(dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    print(f"\n{'='*70}\nFOLD {fold+1}/{len(FOLDS)}  (train {len(train_idx)} / val {len(val_idx)})\n{'='*70}")
    model = CochleogramAST(num_labels=4).to(device)
    opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    criterion = nn.CrossEntropyLoss(weight=cw_t)

    best_score = -1.0; best_state = None; best_epoch = 0
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0; n_b = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item(); n_b += 1
        train_loss /= max(n_b, 1)

        preds, labels, _ = evaluate(model, val_loader)
        m = paper_metrics(preds, labels)
        current_lr = opt.param_groups[0]['lr']
        is_best = m['score'] > best_score
        if is_best:
            best_score = m['score']; best_state = copy.deepcopy(model.state_dict()); best_epoch = epoch + 1
        sched.step()
        marker = '  <- best' if is_best else ''
        print(f'  ep {epoch+1:>2}/{EPOCHS}  train_loss={train_loss:.4f}  '
              f'val Se={m["se"]*100:5.2f} Sp={m["sp"]*100:5.2f} Score={m["score"]*100:5.2f}  '
              f'lr={current_lr:.2e}{marker}')

    # final eval at best checkpoint -- capture softmax probs for ensembling
    model.load_state_dict(best_state)
    preds, labels, probs = evaluate(model, val_loader)
    m = paper_metrics(preds, labels)
    fold_rows.append({'fold': fold + 1, 'best_epoch': best_epoch, **m})
    pooled_preds.extend(preds); pooled_labels.extend(labels)
    per_fold_probs[f'fold{fold+1}_probs']    = probs.astype(np.float32)
    per_fold_labels[f'fold{fold+1}_labels']  = np.asarray(labels, dtype=np.int64)
    per_fold_val_idx[f'fold{fold+1}_val_idx']= np.asarray(val_idx, dtype=np.int64)
    print(f'  FOLD {fold+1} eval: best_ep={best_epoch}  Se={m["se"]*100:5.2f} Sp={m["sp"]*100:5.2f} Score={m["score"]*100:5.2f}')

    del model, opt, sched, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pooled = paper_metrics(pooled_preds, pooled_labels)
se_mean = float(np.mean([r['se']    for r in fold_rows]))
sp_mean = float(np.mean([r['sp']    for r in fold_rows]))
sc_mean = float(np.mean([r['score'] for r in fold_rows]))
sc_std  = float(np.std ([r['score'] for r in fold_rows]))
elapsed = time.time() - t_start

result = {
    'id': 'ast-soft0.25-stratified',
    'config': {'soften_power': SOFTEN_POWER, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
               'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
               'ckpt': 'MIT/ast-finetuned-audioset-10-10-0.4593'},
    'elapsed_sec': elapsed,
    'folds': fold_rows,
    'per_fold_mean': {'se': se_mean, 'sp': sp_mean, 'score': sc_mean, 'score_std': sc_std},
    'pooled_aggregate': pooled,
}
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump(result, f, indent=2, default=str)
np.savez(PREDS_PATH, **per_fold_probs, **per_fold_labels, **per_fold_val_idx)
print(f'\n-> {RESULTS_PATH}')
print(f'-> {PREDS_PATH}')
print(f'\nPER-FOLD MEAN: Se={se_mean*100:.2f}  Sp={sp_mean*100:.2f}  Score={sc_mean*100:.2f}  std={sc_std*100:.2f}')
print(f'POOLED:        Se={pooled["se"]*100:.2f}  Sp={pooled["sp"]*100:.2f}  Score={pooled["score"]*100:.2f}')
print(f'elapsed: {elapsed/60:.1f} min')


class weights (^0.25): [0.763, 0.902, 1.086, 1.249]

FOLD 1/10  (train 6227 / val 671)
  pretrained grid: 12 freq x 101 time = 1212 patches
  target grid:     12 freq x 12 time = 144 patches


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 14142.64it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([4, 768])         
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.

  position embeddings interpolated: 1214 -> 146 tokens
[CochleogramAST] Parameters - total: 85,371,652  trainable: 85,371,652
  ep  1/30  train_loss=0.9849  val Se=57.47 Sp=70.25 Score=63.86  lr=1.25e-05  <- best
  ep  2/30  train_loss=0.8160  val Se=49.68 Sp=75.76 Score=62.72  lr=2.50e-05
  ep  3/30  train_loss=0.6736  val Se=50.97 Sp=76.31 Score=63.64  lr=3.75e-05
  ep  4/30  train_loss=0.5328  val Se=61.04 Sp=59.23 Score=60.13  lr=5.00e-05
  ep  5/30  train_loss=0.3833  val Se=37.34 Sp=85.95 Score=61.64  lr=5.00e-05
  ep  6/30  train_loss=0.2883  val Se=35.71 Sp=75.76 Score=55.74  lr=4.98e-05
  ep  7/30  train_loss=0.1877  val Se=51.95 Sp=69.42 Score=60.68  lr=4.93e-05
  ep  8/30  train_loss=0.1545  val Se=43.18 Sp=73.00 Score=58.09  lr=4.84e-05
  ep  9/30  train_loss=0.1398  val Se=56.17 Sp=62.26 Score=59.21  lr=4.71e-05
  ep 10/30  train_loss=0.1022  val Se=50.32 Sp=72.73 Score=61.53  lr=4.56e-05
  ep 11/30  train_loss=0.1122  val Se=56.49 Sp=65.01 Score=60.75  lr=4.37e-05
  ep 12

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 3960.35it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([4, 768])         
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.S

  position embeddings interpolated: 1214 -> 146 tokens
[CochleogramAST] Parameters - total: 85,371,652  trainable: 85,371,652
  ep  1/30  train_loss=0.9717  val Se=56.99 Sp=66.48 Score=61.74  lr=1.25e-05  <- best
  ep  2/30  train_loss=0.7893  val Se=39.07 Sp=81.59 Score=60.33  lr=2.50e-05
  ep  3/30  train_loss=0.6465  val Se=55.56 Sp=69.23 Score=62.39  lr=3.75e-05  <- best
  ep  4/30  train_loss=0.5362  val Se=55.20 Sp=73.08 Score=64.14  lr=5.00e-05  <- best
  ep  5/30  train_loss=0.3737  val Se=47.67 Sp=70.88 Score=59.27  lr=5.00e-05
  ep  6/30  train_loss=0.2877  val Se=77.78 Sp=42.03 Score=59.91  lr=4.98e-05
  ep  7/30  train_loss=0.1963  val Se=45.16 Sp=84.34 Score=64.75  lr=4.93e-05  <- best
  ep  8/30  train_loss=0.1644  val Se=49.46 Sp=76.92 Score=63.19  lr=4.84e-05
  ep  9/30  train_loss=0.1063  val Se=46.24 Sp=84.89 Score=65.56  lr=4.71e-05  <- best
  ep 10/30  train_loss=0.1191  val Se=49.46 Sp=73.63 Score=61.54  lr=4.56e-05
  ep 11/30  train_loss=0.1140  val Se=55.91 Sp=63

...

  position embeddings interpolated: 1214 -> 146 tokens
[CochleogramAST] Parameters - total: 85,371,652  trainable: 85,371,652
  ep  1/30  train_loss=0.9654  val Se=66.56 Sp=69.78 Score=68.17  lr=1.25e-05  <- best
  ep  2/30  train_loss=0.7832  val Se=63.50 Sp=74.45 Score=68.97  lr=2.50e-05  <- best
  ep  3/30  train_loss=0.6535  val Se=52.45 Sp=72.53 Score=62.49  lr=3.75e-05
  ep  4/30  train_loss=0.5562  val Se=73.93 Sp=57.42 Score=65.67  lr=5.00e-05
  ep  5/30  train_loss=0.4028  val Se=49.69 Sp=76.10 Score=62.90  lr=5.00e-05
  ep  6/30  train_loss=0.3016  val Se=37.12 Sp=86.81 Score=61.96  lr=4.98e-05
  ep  7/30  train_loss=0.1995  val Se=58.90 Sp=70.88 Score=64.89  lr=4.93e-05
  ep  8/30  train_loss=0.1834  val Se=58.28 Sp=69.51 Score=63.89  lr=4.84e-05
  ep  9/30  train_loss=0.1171  val Se=51.53 Sp=77.75 Score=64.64  lr=4.71e-05
  ep 10/30  train_loss=0.1159  val Se=52.45 Sp=71.15 Score=61.80  lr=4.56e-05
  ep 11/30  train_loss=0.0906  val Se=46.93 Sp=79.95 Score=63.44  lr=4.37e-0

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 2828.95it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([4, 768])         
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.S

  position embeddings interpolated: 1214 -> 146 tokens
[CochleogramAST] Parameters - total: 85,371,652  trainable: 85,371,652
  ep  1/30  train_loss=0.9856  val Se=44.91 Sp=81.54 Score=63.23  lr=1.25e-05  <- best
  ep  2/30  train_loss=0.8332  val Se=55.75 Sp=69.97 Score=62.86  lr=2.50e-05
  ep  3/30  train_loss=0.6778  val Se=39.38 Sp=85.12 Score=62.25  lr=3.75e-05
  ep  4/30  train_loss=0.5615  val Se=31.64 Sp=82.64 Score=57.14  lr=5.00e-05
  ep  5/30  train_loss=0.4158  val Se=56.86 Sp=71.07 Score=63.97  lr=5.00e-05  <- best
  ep  6/30  train_loss=0.2903  val Se=42.70 Sp=75.48 Score=59.09  lr=4.98e-05
  ep  7/30  train_loss=0.2120  val Se=40.27 Sp=82.37 Score=61.32  lr=4.93e-05
  ep  8/30  train_loss=0.1571  val Se=38.05 Sp=87.88 Score=62.97  lr=4.84e-05
  ep  9/30  train_loss=0.1355  val Se=38.72 Sp=80.72 Score=59.72  lr=4.71e-05
  ep 10/30  train_loss=0.1013  val Se=37.39 Sp=86.23 Score=61.81  lr=4.56e-05
  ep 11/30  train_loss=0.1194  val Se=46.02 Sp=82.64 Score=64.33  lr=4.37e-0

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 5835.92it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                  
-------------------------------------------------------------+----------+--------------------------------------------------------------------------------------------------
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([4, 768])         
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.S

  position embeddings interpolated: 1214 -> 146 tokens
[CochleogramAST] Parameters - total: 85,371,652  trainable: 85,371,652
  ep  1/30  train_loss=1.0054  val Se=82.49 Sp=74.25 Score=78.37  lr=1.25e-05  <- best
  ep  2/30  train_loss=0.8323  val Se=64.65 Sp=87.67 Score=76.16  lr=2.50e-05
  ep  3/30  train_loss=0.6761  val Se=76.43 Sp=80.00 Score=78.22  lr=3.75e-05
  ep  4/30  train_loss=0.5471  val Se=76.77 Sp=66.30 Score=71.53  lr=5.00e-05
  ep  5/30  train_loss=0.3870  val Se=71.04 Sp=81.37 Score=76.21  lr=5.00e-05
  ep  6/30  train_loss=0.2816  val Se=52.53 Sp=89.86 Score=71.19  lr=4.98e-05
  ep  7/30  train_loss=0.1912  val Se=70.71 Sp=77.81 Score=74.26  lr=4.93e-05
  ep  8/30  train_loss=0.1539  val Se=73.74 Sp=77.26 Score=75.50  lr=4.84e-05
  ep  9/30  train_loss=0.1627  val Se=71.38 Sp=81.10 Score=76.24  lr=4.71e-05
  ep 10/30  train_loss=0.1260  val Se=70.37 Sp=81.37 Score=75.87  lr=4.56e-05
  ep 11/30  train_loss=0.0984  val Se=66.67 Sp=81.92 Score=74.29  lr=4.37e-05
  ep 12

In [ ]:
# --- Compare against previous results ---
print(f"{'config':<30} {'per-fold Sc±std':<18} {'pooled':<8}")
print('-' * 60)
m = result['per_fold_mean']; p = result['pooled_aggregate']
print(f"{result['id']:<30} {m['score']*100:5.2f}±{m['score_std']*100:4.2f}        {p['score']*100:.2f}")
print(f"{'nb18 (best baseline)':<30} {'63.11 ± 3.64':<18} 63.61")
print(f"{'nb16 (MixUp)':<30} {'61.84 ± 6.06':<18} 62.26")
print(f"{'nb12 (KAN)':<30} {'-':<18} 63.16")
print(f"{'literature: AST official':<30} {'-':<18} 59.55")


config                         per-fold Sc±std    pooled  
------------------------------------------------------------
ast-soft0.25-stratified        68.14±4.08        68.02
nb18 (best baseline)           63.11 ± 3.64       63.61
nb16 (MixUp)                   61.84 ± 6.06       62.26
nb12 (KAN)                     -                  63.16
literature: AST official       -                  59.55


: 